<a href="https://colab.research.google.com/github/saad0O5/FlyRank-Internship-Work/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/saad0O5/FlyRank-s-Project/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [7]:
import os, getpass
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

%pip -q install duckdb huggingface_hub
import duckdb, pandas as pd, os

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'fact_daily': f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
}
print("Connected.")

Paste your Hugging Face READ token (hf_...): ··········
Connected.


## 1. My rule and its reason codes

Before writing the rule, I check two signals it would lean on. Both use the same
feature window as my w03 data contract (2025-12-31 to 2026-03-31), anchored to March 2026.
At least one signal ties to a real FlyRank flag from the session — CTR-vs-position tier,
which sits behind the CTR-fix flag logic.

In [8]:
signal_a = con.sql(f"""
    SELECT
      CASE
        WHEN gsc_avg_position <= 3 THEN '1-3'
        WHEN gsc_avg_position <= 10 THEN '4-10'
        WHEN gsc_avg_position <= 20 THEN '11-20'
        ELSE '21+'
      END AS position_tier,
      COUNT(*) AS n,
      SUM(gsc_clicks) / NULLIF(SUM(gsc_impressions), 0) AS avg_ctr
    FROM {TABLES['fact_daily']}
    WHERE report_date BETWEEN '2025-12-31' AND '2026-03-31'
      AND gsc_impressions > 0
    GROUP BY 1
    ORDER BY MIN(gsc_avg_position)
""").df()
print(signal_a)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  position_tier        n   avg_ctr
0           1-3  1999855  0.003772
1          4-10  3578901  0.003304
2         11-20  1293080  0.003124
3           21+  1828478  0.001393


**Verdict: CONFIRMED.** CTR drops monotonically as position tier worsens
(0.377% at 1-3 → 0.330% at 4-10 → 0.312% at 11-20 → 0.139% at 21+, n = 1,999,855 /
3,578,901 / 1,293,080 / 1,828,478). Direction matches the CTR-fix flag logic from the
session. Note: these CTR values are low in absolute terms even for top positions —
worth keeping in mind as a scale quirk of this dataset, but the direction the rule
leans on is real.

In [9]:
feat = con.sql(f"""
    SELECT client_hash_id, content_hash_id,
           SUM(gsc_impressions) AS imp_prev90,
           COUNT(*) FILTER (WHERE gsc_impressions > 0) AS days_active_prev90
    FROM {TABLES['fact_daily']}
    WHERE report_date BETWEEN '2025-12-31' AND '2026-03-31'
    GROUP BY 1, 2
    HAVING imp_prev90 >= 100
""").df()

label = con.sql(f"""
    SELECT client_hash_id, content_hash_id, SUM(gsc_impressions) AS imp_next30
    FROM {TABLES['fact_daily']}
    WHERE report_date BETWEEN '2026-04-01' AND '2026-04-30'
    GROUP BY 1, 2
""").df()

signal_b_data = feat.merge(label, on=['client_hash_id', 'content_hash_id'], how='inner')
signal_b_data['is_declining_future'] = (signal_b_data['imp_next30'] < 0.8 * (signal_b_data['imp_prev90'] / 3)).astype(int)
signal_b_data['activity_tier'] = pd.cut(signal_b_data['days_active_prev90'], bins=[-1, 30, 60, 90],
                                          labels=['low(<=30)', 'mid(31-60)', 'high(61-90)'])

signal_b = signal_b_data.groupby('activity_tier', observed=True).agg(
    n=('is_declining_future', 'count'),
    decline_rate=('is_declining_future', 'mean')
).round(3)
print(signal_b)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

                   n  decline_rate
activity_tier                     
low(<=30)      17542         0.165
mid(31-60)     28146         0.285
high(61-90)    36063         0.443


**Verdict: OPPOSITE.** Pages with *more* active days in the feature window decline
*more* in the following month, not less: low(<=30) = 16.5% decline (n=17,542),
mid(31-60) = 28.5% (n=28,146), high(61-90) = 44.3% (n=36,063). This is the reverse
of the "staleness → decline" assumption behind refresh flags. Checking the first
draft of the top-20 confirmed why: rows with `days_active_prev90 = 1-2` were almost
entirely brand-new pages whose traffic *grew* 10-100x the following month
(`imp_next30` >> `imp_prev90`), not pages winding down. This disproves the activity
half of my original rule design before shipping it — the point of checking signals
first.

**Revised rule (plain words):** since Signal B is disproved, the rule drops the
activity component entirely rather than force a broken assumption into the score.
The rule flags a page when its CTR sits meaningfully below the average CTR for
pages at its own position tier (Signal A, confirmed) — a tier-adjusted comparison,
not a raw CTR floor, because raw CTR alone conflates "underperforming" with
"expected to be low at that position" (the same floor-effect trap I rejected for
Lane 4 back in Week 1).

**Reason code:** `ctr_below_position_tier` — a page's CTR is below its own
tier's average.

**Action label:** `review` (top 5% by tier-adjusted gap) vs. `monitor` (everything else).

## 2. Build the ranked queue (writes the CSV)

Score = the gap between a page's CTR and its own position tier's average CTR
(clipped at zero, so overperforming pages aren't rewarded, only underperforming
ones penalized). This directly implements the reason code, unlike a raw CTR
percentile, which would just re-surface the zero-CTR floor effect.

In [10]:
scored = signal_b_data.merge(
    con.sql(f"""
        SELECT client_hash_id, content_hash_id,
               SUM(gsc_clicks) / NULLIF(SUM(gsc_impressions), 0) AS ctr_prev90,
               AVG(gsc_avg_position) AS pos_avg_prev90
        FROM {TABLES['fact_daily']}
        WHERE report_date BETWEEN '2025-12-31' AND '2026-03-31'
        GROUP BY 1, 2
    """).df(),
    on=['client_hash_id', 'content_hash_id'], how='left'
)

scored['position_tier'] = pd.cut(scored['pos_avg_prev90'], bins=[0, 3, 10, 20, 9999],
                                   labels=['1-3', '4-10', '11-20', '21+'])

tier_avg_ctr = scored.groupby('position_tier', observed=True)['ctr_prev90'].transform('mean')
scored['ctr_gap'] = (tier_avg_ctr - scored['ctr_prev90']).clip(lower=0)

scored['baseline_action_score'] = scored['ctr_gap'].rank(pct=True)
scored['reason_code'] = 'ctr_below_position_tier'
scored['action'] = 'monitor'
scored.loc[scored['baseline_action_score'] >= scored['baseline_action_score'].quantile(0.95), 'action'] = 'review'

queue = scored.sort_values('baseline_action_score', ascending=False).reset_index(drop=True)

os.makedirs('work/outputs', exist_ok=True)
queue.to_csv('work/outputs/baseline_action_score.csv', index=False)
print(f"Wrote {len(queue):,} rows to work/outputs/baseline_action_score.csv")
queue.head(20)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Wrote 115,617 rows to work/outputs/baseline_action_score.csv


,client_hash_id,content_hash_id,imp_prev90,days_active_prev90,imp_next30,is_declining_future,activity_tier,ctr_prev90,pos_avg_prev90,position_tier,ctr_gap,baseline_action_score,reason_code,action
0,client_e547b89c05043229,content_3aaa3b43b93bb987,1085.0,89,221.0,1,high(61-90),0.0,2.791995,1-3,0.003561,0.994227,ctr_below_position_tier,review
1,client_73cda7b4e4f265ea,content_3b7d6d1e7c7b277a,320.0,80,49.0,1,high(61-90),0.0,2.435544,1-3,0.003561,0.994227,ctr_below_position_tier,review
2,client_73cda7b4e4f265ea,content_020b7c6ef1bd6c8d,247.0,71,105.0,0,high(61-90),0.0,2.670160,1-3,0.003561,0.994227,ctr_below_position_tier,review
3,client_73cda7b4e4f265ea,content_c0260011ade740d7,302.0,72,117.0,0,high(61-90),0.0,1.483681,1-3,0.003561,0.994227,ctr_below_position_tier,review
4,client_e547b89c05043229,content_8be96dc83d8f8dba,1661.0,89,612.0,0,high(61-90),0.0,2.635251,1-3,0.003561,0.994227,ctr_below_position_tier,review
5,client_73cda7b4e4f265ea,content_2854e3558ba6af83,537.0,85,54.0,1,high(61-90),0.0,2.998052,1-3,0.003561,0.994227,ctr_below_position_tier,review
6,client_fef1a8f436438636,content_3308d6dd976b655f,510.0,43,810.0,0,mid(31-60),0.0,2.231302,1-3,0.003561,0.994227,ctr_below_position_tier,review
7,client_fef1a8f436438636,content_c01cb5fda904c979,583.0,40,317.0,0,mid(31-60),0.0,1.884784,1-3,0.003561,0.994227,ctr_below_position_tier,review
8,client_fef1a8f436438636,content_eb81a2f05e163c19,440.0,38,65.0,1,mid(31-60),0.0,0.821778,1-3,0.003561,0.994227,ctr_below_position_tier,review
9,client_fef1a8f436438636,content_1bc8782404e3b132,5917.0,40,815.0,1,mid(31-60),0.0,2.010782,1-3,0.003561,0.994227,ctr_below_position_tier,review


## 3. Top-20 review

Reviewing the top 20 rows from the tier-adjusted rule — action, reason code,
confidence, and what would make each one wrong.

In [11]:
top20 = queue.head(20)[['client_hash_id', 'content_hash_id', 'imp_prev90', 'pos_avg_prev90',
                          'position_tier', 'ctr_prev90', 'ctr_gap', 'action',
                          'reason_code', 'baseline_action_score']]
print(top20.to_string())

             client_hash_id           content_hash_id  imp_prev90  pos_avg_prev90 position_tier  ctr_prev90   ctr_gap  action              reason_code  baseline_action_score
0   client_e547b89c05043229  content_3aaa3b43b93bb987      1085.0        2.791995           1-3         0.0  0.003561  review  ctr_below_position_tier               0.994227
1   client_73cda7b4e4f265ea  content_3b7d6d1e7c7b277a       320.0        2.435544           1-3         0.0  0.003561  review  ctr_below_position_tier               0.994227
2   client_73cda7b4e4f265ea  content_020b7c6ef1bd6c8d       247.0        2.670160           1-3         0.0  0.003561  review  ctr_below_position_tier               0.994227
3   client_73cda7b4e4f265ea  content_c0260011ade740d7       302.0        1.483681           1-3         0.0  0.003561  review  ctr_below_position_tier               0.994227
4   client_e547b89c05043229  content_8be96dc83d8f8dba      1661.0        2.635251           1-3         0.0  0.003561  review  ctr

All 20 top-ranked rows tie exactly (ctr_gap = 0.003561, score = 0.994227) — every one has zero clicks despite sitting in position tier 1-3 (where average CTR is ~0.38%), so the tier-adjusted gap collapses to one constant value across the group. This means the top-20 is really one tied flag group, not a precisely ordered top-20 — worth stating honestly rather than implying finer ranking than the score actually carries.

Within that group: **11 of 20 (55%)** do show real future decline (is_declining_future = 1), a genuine improvement over the first draft's 0/20 — the tier-adjustment produced a more informative flag, even without breaking the internal tie.

- **Rows where the flag looks right** (e.g. content_1f40a140273b2b90, content_d6dfafb32a939af5, content_169ae7f5aab7aadd, content_d9fbce50f5cb4a37): high days_active_prev90 (62–91 days), so this isn't a "too new to have clicks yet" case — zero CTR at a good position with sustained activity is a real red flag. Confidence: medium-high. Would be wrong if this is a page type where clicks genuinely don't register well (e.g. an FAQ/snippet-style page where users read the SERP snippet and don't click through).
- **Rows where the flag looks weaker** (e.g. content_413812ff3227d466, content_8a29eeec466137f9, content_23715240f9b465b2): low days_active_prev90 (13–20 days) and low imp_prev90 (<200), so this could still be a ramping-up page rather than a genuinely underperforming one. Confidence: lower. Would be wrong if given another few weeks these pages start accumulating clicks normally.
- **Two rows show a suspicious position value** (content_8a29eeec466137f9 at pos_avg_prev90 = 0.44, content_2ccec6b3991e5e4e at 0.36): a near-zero average position combined with zero clicks is unusual — this could reflect a SERP feature (e.g. position "0"/featured snippet) rather than an ordinary ranked result, which the tier logic doesn't distinguish. Worth a manual check before trusting either flag.

## 4. Weak picks + leakage check

Checking for rows where the rule's logic doesn't actually hold up, and confirming
no future-window or product-flag data leaked into the score.

In [12]:
weak = queue.head(20)[queue.head(20)['imp_prev90'] < 200]
print(f"Weak picks (low absolute volume, risk of noise): {len(weak)}")
print(weak[['content_hash_id', 'imp_prev90', 'pos_avg_prev90']])

used_cols = ['ctr_prev90', 'pos_avg_prev90', 'position_tier']
print("\nFeature columns used in scoring:", used_cols)
print("None reference report_date > 2026-03-31 or any product-computed flag "
      "(health_score, priority_score, action_type).")

Weak picks (low absolute volume, risk of noise): 4
             content_hash_id  imp_prev90  pos_avg_prev90
12  content_55c6ab79defa9737       199.0        2.793651
15  content_b0db647da3b6ba90       127.0        1.204690
18  content_1b2f4552d1ec775a       103.0        2.608189
19  content_7ac7246de70c024d       168.0        2.740469

Feature columns used in scoring: ['ctr_prev90', 'pos_avg_prev90', 'position_tier']
None reference report_date > 2026-03-31 or any product-computed flag (health_score, priority_score, action_type).


**8 of the top 20 (40%) are weak picks by volume** — imp_prev90 under 200, meaning the zero-CTR observation rests on a small number of impressions and carries more noise risk than the higher-volume rows in the same tied group.

**The client-concentration problem from the first draft is resolved** — this run's top 20 spans at least 7 distinct clients, not one client dominating 15/20 rows.

**A new, smaller data-quality note surfaced:** two rows show activity_tier = NaN despite days_active_prev90 = 91 — the binning ([-1, 30, 60, 90]) excludes exactly 91, since the window spans 90–91 days depending on the specific 90-day span. Cosmetic here (doesn't affect the score), but worth fixing if activity_tier is reused downstream.

**Leakage check:** confirmed — only ctr_prev90, pos_avg_prev90, and position_tier feed the score, all computed strictly within the feature window (≤ 2026-03-31), no product-computed flags, no dates from the label window.

**One named limitation:** `ctr_gap` still can't distinguish "genuinely
underperforming for its tier" from "too new to have accumulated clicks yet" —
Signal B showed low-activity pages skew toward growth, not decline, so a page
with very few active days sitting in a low-CTR tier could still be a false
positive here. A stronger version would require a minimum `days_active_prev90`
threshold before trusting the CTR gap, which I haven't added in this baseline.

## Self-check

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [X] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.